# Plot Vegetable Prices by Product
This notebook loads the vegetable prices dataset (category 13) using the repository,
wrangles it into a DataFrame, removes outliers and splits train/test per product,
then plots the concatenated train data with one trace per product using Plotly.

In [14]:
# Setup imports and path
import sys, os
# ensure repo root is on sys.path
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
import pandas as pd
from src.fidzulu.db import oracle_engine
from src.fidzulu.repositories.price_repository import PriceRepository
from src.fidzulu.business.price_wrangler import PriceDataWrangler
from src.fidzulu.business.train_test_splitter import TrainTestSplitter
from src.fidzulu.utils.plotting import price_time_series_figure
print('Imports OK')

Imports OK


In [15]:
# Attempt to fetch vegetable prices (category 13) from the DB
raw_data = None
cache_path = os.path.join('notebooks', 'vegetable_prices_cache.json')
try:
    engine = oracle_engine()
    repo = PriceRepository(engine)
    raw_data = repo.get_prices_by_category(cat_id=13)  # Vegetables
    print('Fetched raw_data from DB; keys:', list(raw_data.keys())[:20])
except Exception as e:
    print('DB access failed:', e)
    # fallback: try loading cached JSON if available
    if os.path.exists(cache_path):
        print('Loading cached dataset from', cache_path)
        raw_data = pd.read_json(cache_path, orient='records')
        # notebooks may have saved a list of records; convert to expected dict if necessary
        if isinstance(raw_data, pd.DataFrame):
            print('Cached file is a DataFrame; cannot convert automatically. Please provide raw_data dict JSON.')
            raw_data = None
    else:
        print('No cache available at', cache_path)

if raw_data is None:
    raise RuntimeError('Unable to obtain vegetable prices dataset from DB or cache. Provide DB access or a cached raw_data dict.')

2026-04-09 11:34:41,439 INFO fidzulu.config: Loaded DBConfig: host=localhost, port=1521, service=xepdb1
2026-04-09 11:34:41,441 INFO src.fidzulu.db: {'event': 'engine_create_attempt', 'db_user': 'fidzulu_pythonmluser', 'host': 'localhost', 'port': 1521, 'service_name': 'xepdb1'}
2026-04-09 11:34:41,441 INFO src.fidzulu.db: Creating Oracle engine with DSN (password redacted from log)
2026-04-09 11:34:41,443 INFO src.fidzulu.db: Oracle engine created successfully
2026-04-09 11:34:41,444 INFO src.fidzulu.repositories.price_repository: {'event': 'query_attempt', 'operation': 'get_prices_by_category', 'query': 'prices_by_category', 'params': {'cat_id': 13}}
2026-04-09 11:34:43,516 WARNING src.fidzulu.repositories.price_repository: {'event': 'anomaly', 'message': 'filtered_invalid_price_rows', 'details': {'category': 13, 'skipped_rows': 1}}


Fetched raw_data from DB; keys: ['CategoryID', 105, 106]


In [16]:
# Convert raw_data (dict as returned by PriceRepository) into a DataFrame via the wrangler
wrangler = PriceDataWrangler(raw_data)
df, feedback = wrangler.wrangle()
print('Wrangled DataFrame shape:', df.shape)
print('Wrangler feedback:', feedback)
df.head()

Wrangled DataFrame shape: (52, 4)
Wrangler feedback: {}


,prod_id,base_price,start_date,end_date
0,105,9.9500,2022-11-01,2022-11-30
1,105,2.5000,2023-01-01,2023-01-31
2,105,2.6500,2023-02-01,2023-02-28
3,105,2.4000,2023-03-01,2023-03-31
4,105,2.7500,2023-04-01,2023-04-30


In [17]:
# Split and remove outliers, obtain concatenated train DataFrame
splitter = TrainTestSplitter(df, test_ratio=0.2)
concatenated_train, train_splits, test_splits = splitter.split_with_median_iqr_concat()
print('Concatenated train shape:', concatenated_train.shape)
list(train_splits.keys())[:10]

[TrainTestSplitter] product 105: train rows before=22, after=21, test rows=5
[TrainTestSplitter] product 106: train rows before=20, after=20, test rows=5
[TrainTestSplitter] processed 2 products, concatenated train shape=(41, 5)
Concatenated train shape: (41, 5)


[105, 106]

In [18]:
# Create and display Plotly figure (one trace per product)
fig = price_time_series_figure(concatenated_train, title='Vegetable Prices - Train Data')
# In Jupyter this will render inline
fig.show()

[plotting] start_date coerced from datetime64[us] to datetime64[us]; base_price coerced from float64 to float64
[plotting] created figure with 2 traces


In [19]:
# Fit double-sinusoid trend per product and summarize results
from src.fidzulu.business.fitting import fit_double_sin_per_product
# train_splits was produced above; run fits on it
fit_results = fit_double_sin_per_product(train_splits)
# show a compact summary for the first 10 products
for pid, r in list(fit_results.items())[:10]:
    if r['success']:
        p = r['params']
        print(f'prod_id={pid}: intercept={p[0]:.3f}, slope={p[1]:.6f}, A1={p[2]:.3f}, freq1={p[3]:.6f}, A2={p[5]:.3f}, freq2={p[6]:.6f}')
    else:
        print(f'prod_id={pid}: fit failed: {r.get("message")}')

# Diagnostic: print small table of train/test rows for products of interest
for pid in (105, 106):
    print('\n--- DIAG product', pid, '---')
    tr = train_splits.get(pid) if 'train_splits' in globals() else None
    te = test_splits.get(pid) if 'test_splits' in globals() else None
    print('train rows:', None if tr is None else len(tr))
    if tr is not None and not tr.empty:
        cols = [c for c in ['start_date','base_price','t'] if c in tr.columns]
        print(tr[cols].to_string(index=False))
    print('test rows:', None if te is None else len(te))
    if te is not None and not te.empty:
        cols = [c for c in ['start_date','base_price','t'] if c in te.columns]
        print(te[cols].to_string(index=False))

prod_id=105: intercept=5.049, slope=-0.005450, A1=3.588, freq1=0.000329, A2=0.016, freq2=0.004961
prod_id=106: intercept=2.688, slope=0.002919, A1=2.119, freq1=0.000158, A2=0.014, freq2=0.004728

--- DIAG product 105 ---
train rows: 21
start_date  base_price   t
2023-01-01      2.5000  61
2023-02-01      2.6500  92
2023-03-01      2.4000 120
2023-04-01      2.7500 151
2023-05-01      2.5500 181
2023-06-01      2.8500 212
2023-07-01      2.6000 242
2023-08-01      2.9500 273
2023-09-01      2.7000 304
2023-10-01      3.0500 334
2023-11-01      2.8000 365
2023-12-01      3.1500 395
2024-01-01      2.9500 426
2024-02-01      3.3000 457
2024-03-01      3.0500 486
2024-04-01      3.4000 517
2024-05-01      3.1500 547
2024-06-01      3.5000 578
2024-07-01      3.2500 608
2024-08-01      3.6000 639
2024-09-01      3.3500 670
test rows: 5
start_date  base_price   t
2024-10-01     10.1500 700
2024-11-01      3.4500 731
2024-11-15    200.0000 745
2024-12-01      3.8000 761
2025-01-01      3.5500

In [20]:
# Plot example: product with at least 6 points and a successful fit
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
matplotlib.use('Agg')
# pick a product id that succeeded
example_pid = None
for pid, r in fit_results.items():
    if r['success'] and r['n'] >= 6:
        example_pid = pid
        break
if example_pid is None:
    print('No suitable product fit available to plot')
else:
    dfp = train_splits[example_pid].sort_values('t')
    t = dfp['t'].to_numpy(dtype=float)
    y = pd.to_numeric(dfp['base_price'], errors='coerce').to_numpy(dtype=float)
    p = fit_results[example_pid]['params']
    y_fit = None
    try:
        from src.fidzulu.business.synodical_regressor import double_sin_trend_model
        y_fit = double_sin_trend_model(t, *p)
    except Exception as exc:
        print('Could not compute model fit for plotting:', exc)
    plt.figure(figsize=(8,4))
    plt.plot(t, y, 'o', label='data')
    if y_fit is not None:
        # overlay continuous model curve
        tt = np.linspace(t.min(), t.max(), 200)
        yy = double_sin_trend_model(tt, *p)
        plt.plot(tt, yy, '-', label='double-sin fit')
    plt.xlabel('t (days since first date)')
    plt.ylabel('base_price')
    plt.title(f'Product {example_pid} fit')
    plt.legend()
    plt.show()

C:\Users\Associate\AppData\Local\Temp\2\ipykernel_7008\1499658340.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
# Plotly: data markers + fitted double-sin model (one product per color)
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
from src.fidzulu.business.synodical_regressor import double_sin_trend_model
# fit_results and train_splits are expected to exist from earlier cells
colors = px.colors.qualitative.Plotly
fig = go.Figure()
prod_ids = sorted([pid for pid in train_splits.keys()])
for i, prod in enumerate(prod_ids):
    dfp = train_splits.get(prod)
    if dfp is None or dfp.empty:
        continue
    dfp = dfp.sort_values('start_date').copy()
    # ensure types
    dfp['start_date'] = pd.to_datetime(dfp['start_date'], errors='coerce')
    dfp['base_price'] = pd.to_numeric(dfp['base_price'], errors='coerce')
    color = colors[i % len(colors)]
    # scatter markers for actual data
    fig.add_trace(go.Scatter(x=dfp['start_date'], y=dfp['base_price'], mode='markers',
                             name=f'{prod} data', marker=dict(color=color), legendgroup=str(prod)))
    # overlay fitted model if available
    r = fit_results.get(prod)
    if r is not None and r.get('success') and r.get('params') is not None:
        p = r['params']
        # dense t grid and corresponding dates using product's first date as origin
        tmin, tmax = float(dfp['t'].min()), float(dfp['t'].max())
        tt = np.linspace(tmin, tmax, 200)
        base = pd.to_datetime(dfp['start_date'].min())
        dates = base + pd.to_timedelta(tt, unit='D')
        yy = double_sin_trend_model(tt, *p)
        fig.add_trace(go.Scatter(x=dates, y=yy, mode='lines',
                                 name=f'{prod} fit', line=dict(color=color, dash='dash'), legendgroup=str(prod)))
# additionally overlay fitted dashed lines for products 105 and 106 (highlight)
for special_pid in [105, 106]:
    r = fit_results.get(special_pid)
    if r is not None and r.get('success') and r.get('params') is not None:
        p = r['params']
        # attempt to get date range from product train split, fallback to concatenated_train
        dfp = train_splits.get(special_pid)
        if dfp is None or dfp.empty:
            if 'concatenated_train' in globals() and not concatenated_train.empty:
                dfp = concatenated_train[concatenated_train['prod_id'] == special_pid]
        if dfp is None or dfp.empty:
            # no date info; skip
            continue
        dfp = dfp.sort_values('start_date').copy()
        dfp['start_date'] = pd.to_datetime(dfp['start_date'], errors='coerce')
        tmin, tmax = float(dfp['t'].min()), float(dfp['t'].max())
        tt = np.linspace(tmin, tmax, 200)
        base = pd.to_datetime(dfp['start_date'].min())
        dates = base + pd.to_timedelta(tt, unit='D')
        # choose color consistent with product assignment if product in prod_ids, else use hash
        if special_pid in prod_ids:
            color = colors[prod_ids.index(special_pid) % len(colors)]
        else:
            color = colors[hash(special_pid) % len(colors)]
        yy = double_sin_trend_model(tt, *p)
        fig.add_trace(go.Scatter(x=dates, y=yy, mode='lines',
                                 name=f'{special_pid} fit', line=dict(color=color, dash='dash', width=3),
                                 legendgroup=str(special_pid)))
# layout tweaks
fig.update_layout(title='Per-product prices with double-sin fits', xaxis_title='start_date', yaxis_title='base_price', width=1000, height=600)
fig.show()

In [22]:
# Compute residuals (actual - fitted) per product and plot them with Plotly
from src.fidzulu.business.synodical_regressor import double_sin_trend_model
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
# Ensure previous variables exist
if 'train_splits' not in globals() or 'fit_results' not in globals():
    raise RuntimeError('run the splitting and fitting cells before computing residuals')
res_counts = {}
for pid, dfp in list(train_splits.items()):
    if dfp is None or dfp.empty:
        continue
    # coerce types
    dfp = dfp.copy()
    if 't' not in dfp.columns:
        # try to compute t if start_date exists
        if 'start_date' in dfp.columns:
            base = pd.to_datetime(dfp['start_date']).min()
            dfp['t'] = (pd.to_datetime(dfp['start_date']) - base).dt.days
        else:
            dfp['t'] = pd.NA
    dfp['t'] = pd.to_numeric(dfp['t'], errors='coerce')
    dfp['base_price'] = pd.to_numeric(dfp['base_price'], errors='coerce')
    r = fit_results.get(pid)
    if r is not None and r.get('success') and r.get('params') is not None:
        p = r['params']
        t_vals = dfp['t'].to_numpy(dtype=float)
        try:
            fitted_vals = double_sin_trend_model(t_vals, *p)
        except Exception:
            fitted_vals = np.full(t_vals.shape, np.nan)
    else:
        fitted_vals = np.full(len(dfp), np.nan)
    dfp['fitted'] = fitted_vals
    dfp['residual'] = dfp['base_price'] - dfp['fitted']
    train_splits[pid] = dfp
    res_counts[pid] = int(np.isfinite(dfp['residual']).sum())
print('Computed residuals for products (sample):', dict(list(res_counts.items())[:10]))
# build a combined DataFrame for plotting
res_dfs = []
for pid, dfp in train_splits.items():
    if dfp is None or dfp.empty:
        continue
    if 'residual' in dfp.columns:
        dfp2 = dfp[['prod_id','start_date','t','base_price','fitted','residual']].copy()
        res_dfs.append(dfp2)
if res_dfs:
    residuals_df = pd.concat(res_dfs, ignore_index=True)
    residuals_df['start_date'] = pd.to_datetime(residuals_df['start_date'], errors='coerce')
else:
    residuals_df = pd.DataFrame(columns=['prod_id','start_date','t','base_price','fitted','residual'])
# Plot with Plotly: one trace per product
fig = go.Figure()
colors = px.colors.qualitative.Plotly
prod_ids = sorted(residuals_df['prod_id'].dropna().unique())
for i, pid in enumerate(prod_ids):
    grp = residuals_df[residuals_df['prod_id'] == pid].sort_values('start_date')
    if grp.empty:
        continue
    color = colors[i % len(colors)]
    fig.add_trace(go.Scatter(x=grp['start_date'], y=grp['residual'], mode='lines+markers',
                         name=str(pid), line=dict(color=color), marker=dict(color=color)))
# add zero line
if not residuals_df.empty:
    fig.add_trace(go.Scatter(x=[residuals_df['start_date'].min(), residuals_df['start_date'].max()], y=[0,0], mode='lines', line=dict(color='black', dash='dot'), showlegend=False))
fig.update_layout(title='Residuals over time (actual - fitted)', xaxis_title='start_date', yaxis_title='residual', width=1000, height=600)
fig.show()

Computed residuals for products (sample): {105: 21, 106: 20}


In [23]:
# 180-day forecast using fitted models
from src.fidzulu.business.synodical_regressor import double_sin_trend_model
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

if 'fit_results' not in globals() or 'train_splits' not in globals():
    raise RuntimeError('run the splitting and fitting cells before forecasting')

forecasts = []
for pid, r in fit_results.items():
    if not (r and r.get('success') and r.get('params') is not None):
        continue
    dfp = train_splits.get(pid)
    if dfp is None or dfp.empty:
        # try concatenated_train as fallback
        if 'concatenated_train' in globals() and not concatenated_train.empty:
            dfp = concatenated_train[concatenated_train['prod_id'] == pid]
    if dfp is None or dfp.empty:
        continue
    dfp = dfp.sort_values('start_date').copy()
    dfp['start_date'] = pd.to_datetime(dfp['start_date'], errors='coerce')
    dfp['t'] = pd.to_numeric(dfp['t'], errors='coerce')
    last_t = float(dfp['t'].max())
    last_date = pd.to_datetime(dfp['start_date'].max())
    days_ahead = np.arange(1, 181)
    t_future = last_t + days_ahead
    dates_future = last_date + pd.to_timedelta(days_ahead, unit='D')
    p = r['params']
    y_future = double_sin_trend_model(t_future, *p)
    df_future = pd.DataFrame({'prod_id': pid, 'start_date': dates_future, 't': t_future, 'predicted': y_future})
    forecasts.append(df_future)
    # store forecast per product inside fit_results for later inspection
    r['forecast'] = df_future

if forecasts:
    forecast_df = pd.concat(forecasts, ignore_index=True)
    forecast_df['start_date'] = pd.to_datetime(forecast_df['start_date'])
else:
    forecast_df = pd.DataFrame(columns=['prod_id', 'start_date', 't', 'predicted'])

print('Generated forecast rows:', len(forecast_df))

# Plot forecasts (one dashed line per product)
fig = go.Figure()
colors = px.colors.qualitative.Plotly
prod_ids = sorted(forecast_df['prod_id'].dropna().unique())
for i, pid in enumerate(prod_ids):
    grp = forecast_df[forecast_df['prod_id'] == pid].sort_values('start_date')
    if grp.empty:
        continue
    color = colors[i % len(colors)]
    fig.add_trace(go.Scatter(x=grp['start_date'], y=grp['predicted'], mode='lines',
                             name=f'{pid} forecast', line=dict(color=color, dash='dash')))

fig.update_layout(title='180-day forecast per product', xaxis_title='date', yaxis_title='predicted_price', width=1000, height=600)
fig.show()

Generated forecast rows: 360


In [24]:
# Summary: unpack fitted parameters and detect 1 vs 2 seasonal components
import numpy as np
import pandas as pd

if 'fit_results' not in globals():
    raise RuntimeError('run the fitting cell before this summary')

def _effective_second_component(A1, A2, rel_thresh=0.1, abs_thresh=1e-3):
    # Consider second component present if amplitude A2 is both finite and meaningfully large
    try:
        return np.isfinite(A2) and (A2 > abs_thresh) and (A2 / A1 > rel_thresh)
    except Exception:
        return False

for pid in sorted(fit_results.keys()):
    r = fit_results[pid]
    if not r.get('success'):
        print(f'prod_id={pid}: fit failed: {r.get("message")}')
        continue
    p = r.get('params')
    if p is None:
        print(f'prod_id={pid}: no parameters')
        continue
    intercept, slope = float(p[0]), float(p[1])
    A1, f1, ph1 = float(p[2]), float(p[3]), float(p[4])
    A2, f2, ph2 = float(p[5]), float(p[6]), float(p[7])

    has_second = _effective_second_component(A1, A2)
    comp_count = 2 if has_second else 1

    # residual RMS if available in train_splits
    rms = None
    dfp = globals().get('train_splits', {}).get(pid)
    if dfp is not None and hasattr(dfp, '__len__') and len(dfp) > 0 and 'residual' in dfp.columns:
        resid = pd.to_numeric(dfp['residual'], errors='coerce').to_numpy(dtype=float)
        mask = np.isfinite(resid)
        if mask.sum() > 0:
            rms = float(np.sqrt(np.nanmean(resid[mask]**2)))

    period1 = (1.0 / f1) if f1 > 0 else float('inf')
    period2 = (1.0 / f2) if f2 > 0 else float('inf')

    print('-'*72)
    print(f'Product {pid}: components={comp_count}  intercept={intercept:.3f}  slope={slope:.6f}  RMS={rms if rms is not None else "?"}  n={r.get("n", "?")}')
    print(f'  Component 1: A={A1:.3f}  freq={f1:.6f} cycles/day  period={period1:.1f} days  phase={ph1:.3f}')
    if has_second:
        print(f'  Component 2: A={A2:.3f}  freq={f2:.6f} cycles/day  period={period2:.1f} days  phase={ph2:.3f}')

print('-'*72)

------------------------------------------------------------------------
Product 105: components=1  intercept=5.049  slope=-0.005450  RMS=0.1408736073465559  n=21
  Component 1: A=3.588  freq=0.000329 cycles/day  period=3040.5 days  phase=5.504
------------------------------------------------------------------------
Product 106: components=1  intercept=2.688  slope=0.002919  RMS=0.11975391864342352  n=20
  Component 1: A=2.119  freq=0.000158 cycles/day  period=6323.9 days  phase=3.735
------------------------------------------------------------------------


In [25]:
# Evaluate model on test set: baseline vs fitted model (MAE, MSE, RMSE per product)
import numpy as np
import pandas as pd
from src.fidzulu.business.synodical_regressor import double_sin_trend_model

if 'test_splits' not in globals():
    raise RuntimeError('run the splitting cell to produce `test_splits` before running evaluation')

def _mae(y, yhat):
    mask = np.isfinite(y) & np.isfinite(yhat)
    if not mask.any():
        return float('nan')
    return float(np.mean(np.abs(y[mask] - yhat[mask])))
def _mse(y, yhat):
    mask = np.isfinite(y) & np.isfinite(yhat)
    if not mask.any():
        return float('nan')
    return float(np.mean((y[mask] - yhat[mask])**2))
def _rmse(y, yhat):
    v = _mse(y, yhat)
    return float(np.sqrt(v)) if np.isfinite(v) else float('nan')

rows = []
for pid, df_test in sorted(test_splits.items()):
    if df_test is None or getattr(df_test, 'empty', False):
        continue
    df_test = df_test.copy()
    df_test['start_date'] = pd.to_datetime(df_test['start_date'], errors='coerce')
    if 't' not in df_test.columns:
        if 'start_date' in df_test.columns:
            # compute t relative to the training base date for this product if available
            train_df = globals().get('train_splits', {}).get(pid)
            if train_df is not None and not train_df.empty and 't' in train_df.columns:
                # use the same origin as training data
                train_base = pd.to_datetime(train_df['start_date']).min()
            else:
                train_base = pd.to_datetime(df_test['start_date']).min()
            df_test['t'] = (pd.to_datetime(df_test['start_date']) - train_base).dt.days
        else:
            df_test['t'] = float('nan')
    df_test['t'] = pd.to_numeric(df_test['t'], errors='coerce')
    df_test['base_price'] = pd.to_numeric(df_test['base_price'], errors='coerce')
    y_true = df_test['base_price'].to_numpy(dtype=float)
    n_test = int(np.isfinite(y_true).sum())
    # baseline: last observed training value for this product (persistence)
    baseline_pred = np.full(len(df_test), np.nan, dtype=float)
    if 'train_splits' in globals() and pid in train_splits and train_splits[pid] is not None and not train_splits[pid].empty:
        df_train = train_splits[pid].sort_values('start_date').copy()
        df_train['base_price'] = pd.to_numeric(df_train['base_price'], errors='coerce')
        finite_train = df_train['base_price'][np.isfinite(df_train['base_price'].to_numpy(dtype=float))]
        if len(finite_train) > 0:
            baseline_pred = np.full(len(df_test), float(finite_train.iloc[-1]), dtype=float)
    # fallback: train median if needed
    if not np.isfinite(baseline_pred).any():
        if 'train_splits' in globals() and pid in train_splits and train_splits[pid] is not None and not train_splits[pid].empty:
            med = pd.to_numeric(train_splits[pid]['base_price'], errors='coerce').median()
            if np.isfinite(med):
                baseline_pred = np.full(len(df_test), float(med), dtype=float)

    baseline_mae = _mae(y_true, baseline_pred)
    baseline_mse = _mse(y_true, baseline_pred)
    baseline_rmse = _rmse(y_true, baseline_pred)

    # model predictions if available
    model_pred = np.full(len(df_test), np.nan, dtype=float)
    model_success = False
    model_msg = None
    r = globals().get('fit_results', {}).get(pid) if 'fit_results' in globals() else None
    if r is not None and r.get('success') and r.get('params') is not None:
        try:
            params = r['params']
            tvals = df_test['t'].to_numpy(dtype=float)
            model_pred = double_sin_trend_model(tvals, *params)
            model_success = True
        except Exception as exc:
            model_msg = str(exc)
    else:
        if r is not None:
            model_msg = r.get('message')

    model_mae = _mae(y_true, model_pred)
    model_mse = _mse(y_true, model_pred)
    model_rmse = _rmse(y_true, model_pred)

    rows.append({
        'prod_id': pid, 'n_test': n_test,
        'baseline_mae': baseline_mae, 'baseline_mse': baseline_mse, 'baseline_rmse': baseline_rmse,
        'model_mae': model_mae, 'model_mse': model_mse, 'model_rmse': model_rmse,
        'model_success': bool(model_success), 'model_message': model_msg
    })

metrics_df = pd.DataFrame(rows) if rows else pd.DataFrame(columns=['prod_id'])
pd.options.display.float_format = '{:0.4f}'.format
print('Per-product evaluation (baseline vs fitted model):')
display_cols = ['prod_id','n_test','baseline_mae','baseline_rmse','model_mae','model_rmse','model_success']
if not metrics_df.empty:
    print(metrics_df[display_cols].to_string(index=False))
else:
    print('No test rows found to evaluate')

Per-product evaluation (baseline vs fitted model):
 prod_id  n_test  baseline_mae  baseline_rmse  model_mae  model_rmse  model_success
     105       5       40.8400        87.9974    40.7411     87.9517           True
     106       5        0.1200         0.1414     0.1450      0.1728           True


In [26]:
# Diagnostic: print train/test rows for products 105 and 106
import pandas as pd
for pid in (105, 106):
    print('\n---- PRODUCT', pid, '----')
    tr = train_splits.get(pid) if 'train_splits' in globals() else None
    te = test_splits.get(pid) if 'test_splits' in globals() else None
    print('TRAIN rows:', None if tr is None else len(tr))
    if tr is not None and not tr.empty:
        display_cols = [c for c in ['start_date','base_price','t'] if c in tr.columns]
        print(tr[display_cols].to_string(index=False))
    print('TEST rows:', None if te is None else len(te))
    if te is not None and not te.empty:
        display_cols = [c for c in ['start_date','base_price','t'] if c in te.columns]
        print(te[display_cols].to_string(index=False))


---- PRODUCT 105 ----
TRAIN rows: 21
start_date  base_price   t
2023-01-01      2.5000  61
2023-02-01      2.6500  92
2023-03-01      2.4000 120
2023-04-01      2.7500 151
2023-05-01      2.5500 181
2023-06-01      2.8500 212
2023-07-01      2.6000 242
2023-08-01      2.9500 273
2023-09-01      2.7000 304
2023-10-01      3.0500 334
2023-11-01      2.8000 365
2023-12-01      3.1500 395
2024-01-01      2.9500 426
2024-02-01      3.3000 457
2024-03-01      3.0500 486
2024-04-01      3.4000 517
2024-05-01      3.1500 547
2024-06-01      3.5000 578
2024-07-01      3.2500 608
2024-08-01      3.6000 639
2024-09-01      3.3500 670
TEST rows: 5
start_date  base_price   t
2024-10-01     10.1500 700
2024-11-01      3.4500 731
2024-11-15    200.0000 745
2024-12-01      3.8000 761
2025-01-01      3.5500 792

---- PRODUCT 106 ----
TRAIN rows: 20
start_date  base_price   t
2023-01-05      1.5000   0
2023-02-01      1.6500  27
2023-03-01      1.4000  55
2023-04-01      1.7000  86
2023-05-01      1.55